## Conexión AstraDB y Spark session

### Prerrequisito

- `ASTRA_CLIENT_ID` y `ASTRA_CLIENT_SECRET` definidas como secret variables en colab.
- AstraBD `secure-connect-cloud-analytics.zip` guardado en la carpeta raiz del proyecto.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/big-data-final"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Instalar Java 17
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

# Instalar PySpark y driver
!pip install pyspark==3.5.0 cassandra-driver -q

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

print("Dependencias instaladas.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 20.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires pyspark[connect]~=3.5.1, but you have pyspark 3.5.0 which is incompatible.
Dependencias instaladas.


In [19]:
# Rutas del Data Lake
LANDING_PATH = f"{PROJECT_ROOT}/datalake/landing"
BRONZE_PATH = f"{PROJECT_ROOT}/datalake/bronze"
SILVER_PATH = f"{PROJECT_ROOT}/datalake/silver"
GOLD_PATH = f"{PROJECT_ROOT}/datalake/gold"
CHECKPOINT_PATH = f"{PROJECT_ROOT}/datalake/checkpoints"
QUARANTINE_PATH = f"{PROJECT_ROOT}/datalake/quarantine"

# Crear estructura de directorios si no existe
os.makedirs(LANDING_PATH, exist_ok=True)
os.makedirs(BRONZE_PATH, exist_ok=True)
os.makedirs(SILVER_PATH, exist_ok=True)
os.makedirs(GOLD_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)
os.makedirs(QUARANTINE_PATH, exist_ok=True)

print(f"Directorios configurados en: {PROJECT_ROOT}")

Directorios configurados en: /content/drive/MyDrive/big-data-final


In [20]:
import shutil
from google.colab import userdata

# Credenciales de AstraDB
ASTRA_CLIENT_ID = userdata.get('ASTRA_CLIENT_ID')
ASTRA_CLIENT_SECRET = userdata.get('ASTRA_CLIENT_SECRET')

# Ruta al Secure Connect Bundle (SCB)
SCB_PATH = f"{PROJECT_ROOT}/secure-connect-cloud-analytics.zip"

if os.path.exists(SCB_PATH):
    print(f"Secure Connect Bundle encontrado en: {SCB_PATH}")
    abs_path = os.path.abspath(SCB_PATH)
    SCB_URI = f"file://{abs_path}"
else:
    print(f"No se encuentra el Secure Connect Bundle en {SCB_PATH}")
    raise FileNotFoundError(f"Sube el secure-connect-bundle.zip a {PROJECT_ROOT}")

if 'spark' in locals():
    spark.stop()

Secure Connect Bundle encontrado en: /content/drive/MyDrive/big-data-final/secure-connect-cloud-analytics.zip


In [21]:
# SparkSession
# Configuramos Spark para que descargue automáticamente el conector de Cassandra y utilice el SCB.
from pyspark.sql import SparkSession

SPARK_PACKAGES = "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0"

spark = SparkSession.builder \
    .appName("CloudProviderAnalytics_Colab") \
    .master("local[*]") \
    .config("spark.jars.packages", SPARK_PACKAGES) \
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions") \
    .config("spark.sql.catalog.myCatalog", "com.datastax.spark.connector.datasource.CassandraCatalog") \
    .config("spark.cassandra.connection.config.cloud.path", SCB_URI) \
    .config("spark.cassandra.auth.username", ASTRA_CLIENT_ID) \
    .config("spark.cassandra.auth.password", ASTRA_CLIENT_SECRET) \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(f"Spark Session iniciada. Versión: {spark.version}")

Spark Session iniciada. Versión: 3.5.0


In [22]:
# Smoke test para verificar la conexión

# 1. Test Spark Local
try:
    print("Test 1: Spark Local DataFrame...")
    spark.range(3).show()
    print("✅ Spark local OK.")
except Exception as e:
    print(f"❌ Error Spark local: {e}")

# 2. Test Conectividad AstraDB
print("\nTest 2: Conexión a AstraDB...")
try:
    # Usamos el catálogo 'myCatalog' que definimos en la configuración
    # Esto le pide a AstraDB la lista de Keyspaces visibles
    spark.sql("SHOW NAMESPACES IN myCatalog").show()
    print("✅ Verifica la existencia de 'cloud_analytics'")
    spark.sql("DESCRIBE NAMESPACE myCatalog.cloud_analytics").show(truncate=False)

except Exception as e:
    print(f"❌ Error al listar bases de datos: {e}")

Test 1: Spark Local DataFrame...
+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+

✅ Spark local OK.

Test 2: Conexión a AstraDB...
+------------------+
|         namespace|
+------------------+
|   cloud_analytics|
|data_endpoint_auth|
|      datastax_sla|
+------------------+

✅ Verifica la existencia de 'cloud_analytics'
+--------------+---------------+
|info_name     |info_value     |
+--------------+---------------+
|Catalog Name  |myCatalog      |
|Namespace Name|cloud_analytics|
+--------------+---------------+



## Creacion de Tablas Cassandra

In [23]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

print("Conectando a AstraDB para crear tablas...")
cloud_config = {'secure_connect_bundle': SCB_PATH}
auth_provider = PlainTextAuthProvider(ASTRA_CLIENT_ID, ASTRA_CLIENT_SECRET)
cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
session = cluster.connect()

# FinOps Daily Usage
print("Creando tabla: finops_daily_usage...")
session.execute("""
CREATE TABLE IF NOT EXISTS cloud_analytics.finops_daily_usage (
    org_id TEXT,
    usage_date DATE,
    service TEXT,
    total_cost_usd DECIMAL,
    total_requests BIGINT,
    total_tokens BIGINT,
    total_carbon_kg DECIMAL,
    is_anomaly BOOLEAN,
    anomaly_score DOUBLE,
    PRIMARY KEY ((org_id), usage_date, service)
) WITH CLUSTERING ORDER BY (usage_date DESC, service ASC)
  AND default_time_to_live = 7776000
""")
print("- finops_daily_usage creada\n")

# Org Service Cost Summary
print("Creando tabla: org_service_cost_summary...")
session.execute("""
CREATE TABLE IF NOT EXISTS cloud_analytics.org_service_cost_summary (
    org_id TEXT,
    summary_date DATE,
    service TEXT,
    cost_14d DECIMAL,
    cost_7d DECIMAL,
    cost_1d DECIMAL,
    rank_14d INT,
    PRIMARY KEY ((org_id, summary_date), rank_14d, service)
) WITH CLUSTERING ORDER BY (rank_14d ASC)
""")
print("- org_service_cost_summary creada\n")

# Support Tickets Summary
print("Creando tabla: support_tickets_summary...")
session.execute("""
CREATE TABLE IF NOT EXISTS cloud_analytics.support_tickets_summary (
    org_id TEXT,
    ticket_date DATE,
    severity TEXT,
    total_tickets INT,
    avg_csat DECIMAL,
    sla_breaches_count INT,
    PRIMARY KEY ((org_id), ticket_date, severity)
) WITH CLUSTERING ORDER BY (ticket_date DESC)
""")
print("- support_tickets_summary creada\n")

# Revenue Monthly
print("Creando tabla: revenue_monthly...")
session.execute("""
CREATE TABLE IF NOT EXISTS cloud_analytics.revenue_monthly (
    org_id TEXT,
    billing_month DATE,
    currency TEXT,
    total_due_local DECIMAL,
    credits_applied DECIMAL,
    tax_amount DECIMAL,
    net_revenue DECIMAL,
    PRIMARY KEY ((org_id), billing_month)
) WITH CLUSTERING ORDER BY (billing_month DESC)
""")
print("- revenue_monthly creada\n")

# GenAI Usage Daily
print("Creando tabla: genai_usage_daily...")
session.execute("""
CREATE TABLE IF NOT EXISTS cloud_analytics.genai_usage_daily (
    org_id TEXT,
    usage_date DATE,
    total_tokens BIGINT,
    total_requests INT,
    total_cost_usd DECIMAL,
    avg_tokens_per_request INT,
    PRIMARY KEY ((org_id), usage_date)
) WITH CLUSTERING ORDER BY (usage_date DESC)
""")
print("- genai_usage_daily creada\n")

# Carbon Footprint Daily
print("Creando tabla: carbon_footprint_daily...")
session.execute("""
CREATE TABLE IF NOT EXISTS cloud_analytics.carbon_footprint_daily (
    org_id TEXT,
    usage_date DATE,
    total_carbon_kg DECIMAL,
    events_with_carbon_data INT,
    PRIMARY KEY ((org_id), usage_date)
) WITH CLUSTERING ORDER BY (usage_date DESC)
""")
print("- carbon_footprint_daily creada (BONUS)\n")


print("Verificando tablas en keyspace cloud_analytics:")
rows = session.execute("SELECT table_name FROM system_schema.tables WHERE keyspace_name='cloud_analytics'")
for row in rows:
    print(f"  - {row.table_name}")

cluster.shutdown()
print("\nTodas las tablas creadas exitosamente!")

Conectando a AstraDB para crear tablas...


Creando tabla: finops_daily_usage...
- finops_daily_usage creada

Creando tabla: org_service_cost_summary...
- org_service_cost_summary creada

Creando tabla: support_tickets_summary...
- support_tickets_summary creada

Creando tabla: revenue_monthly...
- revenue_monthly creada

Creando tabla: genai_usage_daily...
- genai_usage_daily creada

Creando tabla: carbon_footprint_daily...
- carbon_footprint_daily creada (BONUS)

Verificando tablas en keyspace cloud_analytics:
  - carbon_footprint_daily
  - finops_daily_usage
  - genai_usage_daily
  - org_service_cost_summary
  - revenue_monthly
  - support_tickets_summary

Todas las tablas creadas exitosamente!


## Ingest Batch (Landing -> Bronze)

In [24]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, input_file_name

print("Ingest Batch a Bronze...")

# DEFINICIÓN DE ESQUEMAS
# Definimos los tipos manualmente para asegurar calidad desde el inicio.
# Esto evita que Spark adivine mal (ej. tratar un ID numérico como entero cuando debería ser string)

schemas = {
    "customers_orgs": StructType([
        StructField("org_id", StringType(), True),
        StructField("org_name", StringType(), True),
        StructField("industry", StringType(), True),
        StructField("hq_region", StringType(), True),
        StructField("plan_tier", StringType(), True),
        StructField("is_enterprise", BooleanType(), True),
        StructField("signup_date", DateType(), True),
        StructField("sales_rep", StringType(), True),
        StructField("lifecycle_stage", StringType(), True),
        StructField("marketing_source", StringType(), True),
        StructField("nps_score", DoubleType(), True)
    ]),
    "users": StructType([
        StructField("user_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("email", StringType(), True),
        StructField("role", StringType(), True),
        StructField("active", BooleanType(), True),
        StructField("created_at", DateType(), True),
        StructField("last_login", DateType(), True)
    ]),
    "resources": StructType([
        StructField("resource_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("service", StringType(), True),
        StructField("region", StringType(), True),
        StructField("created_at", DateType(), True),
        StructField("state", StringType(), True),
        StructField("tags_json", StringType(), True)
    ]),
  "billing_monthly": StructType([
        StructField("invoice_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("month", DateType(), True),
        StructField("subtotal", DoubleType(), True),
        StructField("credits", DoubleType(), True),
        StructField("taxes", DoubleType(), True),
        StructField("currency", StringType(), True),
        StructField("exchange_rate_to_usd", DoubleType(), True)
    ]),
    "support_tickets": StructType([
        StructField("ticket_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("category", StringType(), True),
        StructField("severity", StringType(), True),
        StructField("created_at", DateType(), True),
        StructField("resolved_at", DateType(), True),
        StructField("csat", DoubleType(), True),
        StructField("sla_breached", BooleanType(), True)
    ]),
    "marketing_touches": StructType([
        StructField("touch_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("campaign", StringType(), True),
        StructField("channel", StringType(), True),
        StructField("timestamp", DateType(), True),
        StructField("clicked", BooleanType(), True),
        StructField("converted", BooleanType(), True)
    ]),
    "nps_surveys": StructType([
        StructField("org_id", StringType(), True),
        StructField("survey_date", DateType(), True),
        StructField("nps_score", DoubleType(), True),
        StructField("comment", StringType(), True)
    ])
}

# FUNCIÓN DE INGEST ESTÁNDAR
def ingest_batch_file(file_name, schema):
    source_path = f"{LANDING_PATH}/{file_name}.csv"
    dest_path = f"{BRONZE_PATH}/{file_name}"

    print(f"Procesando: {file_name}...")
    try:
        # mode="PERMISSIVE": Si una fila está muy mal formada, pone nulls pero no rompe el proceso
        df = spark.read.csv(source_path, header=True, schema=schema, mode="PERMISSIVE")

        # Agregar marcas de ingesta (Requisito Bronze)
        df_bronze = df \
            .withColumn("ingest_ts", current_timestamp()) \
            .withColumn("source_file", input_file_name())

        # Escribir a Parquet
        df_bronze.write.mode("overwrite").parquet(dest_path)

        count = df_bronze.count()
        print(f"Guardado en Bronze: {dest_path}")
        print(f"Registros procesados: {count}")

    except Exception as e:
        print(f"Error crítico en {file_name}: {e}")

# EJECUCIÓN DEL PIPELINE BATCH
for name, schema in schemas.items():
    ingest_batch_file(name, schema)

print("\nCapa Bronze Batch lista.")

Ingest Batch a Bronze...
Procesando: customers_orgs...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/customers_orgs
Registros procesados: 80
Procesando: users...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/users
Registros procesados: 800
Procesando: resources...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/resources
Registros procesados: 400
Procesando: billing_monthly...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/billing_monthly
Registros procesados: 240
Procesando: support_tickets...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/support_tickets
Registros procesados: 1000
Procesando: marketing_touches...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/marketing_touches
Registros procesados: 1500
Procesando: nps_surveys...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/nps_surveys
Registros

In [25]:
print("Verificando tabla 'resources' en Bronze:")
df_check = spark.read.parquet(f"{BRONZE_PATH}/resources")
df_check.show(5)
df_check.printSchema()

Verificando tabla 'resources' en Bronze:
+------------+------------+----------+----------+----------+-------+----------------+--------------------+--------------------+
| resource_id|      org_id|   service|    region|created_at|  state|       tags_json|           ingest_ts|         source_file|
+------------+------------+----------+----------+----------+-------+----------------+--------------------+--------------------+
|res_eubfn9kr|org_pnsm43d8|   compute|   sa-east|2025-08-14|running|"[""env:prod""]"|2025-12-04 17:46:...|file:///content/d...|
|res_fvb66h3r|org_i7p5tb94|  database|  ap-south|2025-06-05|stopped|            NULL|2025-12-04 17:46:...|file:///content/d...|
|res_cbrlqmn4|org_d14ve92m|   storage|eu-central|2025-08-10|running|  "[""env:prod""|2025-12-04 17:46:...|file:///content/d...|
|res_ew1yf0dw|org_pja1wj0t|networking|   us-west|2025-06-02|running|"[""pii:true""]"|2025-12-04 17:46:...|file:///content/d...|
|res_n6mbypjd|org_pja1wj0t|   storage|   us-west|2025-06-05|run

### Ingest Streaming (Landing -> Bronze)

Los archivos JSONL en `usage_events_stream/` simulan un flujo continuo de eventos. Tienes un reto clave mencionado en el enunciado: Evolución de Esquema.

Al principio, los eventos tienen un esquema V1.

A partir de cierta fecha (~2025-07-18), aparece la `schema_version=2` con campos nuevos: `genai_tokens` y `carbon_kg`.

In [26]:
# Para que el stream no falle cuando aparezcan los campos nuevos,
# definimos un superset, que inicialmente Spark llenara con null

print("Ingest Streaming...")

# ===== PARA DEBUG =====
# Borramos los checkpoints y datos anteriores para evitar conflictos de esquema
if os.path.exists(f"{CHECKPOINT_PATH}/bronze_events"):
    shutil.rmtree(f"{CHECKPOINT_PATH}/bronze_events")
if os.path.exists(f"{BRONZE_PATH}/usage_events"):
    shutil.rmtree(f"{BRONZE_PATH}/usage_events")
print("Checkpoints y datos anteriores eliminados.")
# ======================

# DEFINIR ESQUEMA UNIFICADO (V1 + V2)
# Si un JSON no tiene algun campo, Spark le pone null.
json_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("org_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("resource_id", StringType(), True),
    StructField("action", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("unit", StringType(), True),
    StructField("cost_usd_increment", DoubleType(), True),
    StructField("schema_version", StringType(), True),
    # Campos nuevos de V2 (apareceran como null en V1)
    StructField("genai_tokens", LongType(), True),
    StructField("carbon_kg", DoubleType(), True)
])

# CONFIGURAR EL READER
print("Configurando lectura de stream desde Landing...")

df_stream_raw = spark.readStream \
    .format("json") \
    .schema(json_schema) \
    .option("maxFilesPerTrigger", 10) \
    .load(f"{LANDING_PATH}/usage_events_stream") #

# LÓGICA DE DEDUPE Y METADATA
# Requisito: Deduplicar por event_id y manejar late data
df_stream_bronze = df_stream_raw \
    .withColumnRenamed("timestamp", "event_ts") \
    .withColumn("ingest_ts", current_timestamp()) \
    .withColumn("source_file", input_file_name()) \
    .withWatermark("event_ts", "2 hours") \
    .dropDuplicates(["event_id", "event_ts"])

# CONFIGURAR EL WRITER (A BRONZE PARQUET)
# Guardamos en Parquet particionado por fecha para optimizar lecturas futuras
# Usamos checkpointing en Drive para tolerancia a fallos
bronze_stream_query = df_stream_bronze \
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/bronze_events") \
    .option("path", f"{BRONZE_PATH}/usage_events") \
    .partitionBy("service") \
    .trigger(availableNow=True) \
    .start()

print("Stream iniciado con trigger='availableNow'...")
print("- Esto procesará TODOS los archivos históricos disponibles.")
print("- Espera a que termine el proceso...")

# Esperar a que termine de procesar los archivos actuales
bronze_stream_query.awaitTermination()

print("Ingesta Streaming completada (Modo Batch Inicial).")

Ingest Streaming...
Checkpoints y datos anteriores eliminados.
Configurando lectura de stream desde Landing...
Stream iniciado con trigger='availableNow'...
- Esto procesará TODOS los archivos históricos disponibles.
- Espera a que termine el proceso...
Ingesta Streaming completada (Modo Batch Inicial).


In [27]:
from pyspark.sql.functions import col

print("Inspeccionando capa Bronze (Streaming Events):")

try:
    path_events = f"{BRONZE_PATH}/usage_events"
    df_events = spark.read.parquet(path_events)

    total_events = df_events.count()
    print(f"Total de eventos ingestados: {total_events}")
    print("Muestra de datos (incluyendo columnas v2):")
    df_events.select("event_ts", "service", "cost_usd_increment", "genai_tokens", "carbon_kg").show(10)
    print("Servicios encontrados (Particiones):")
    df_events.select("service").distinct().show()

except Exception as e:
    print(f"Aún no hay datos o ruta incorrecta: {e}")

Inspeccionando capa Bronze (Streaming Events):
Total de eventos ingestados: 7245
Muestra de datos (incluyendo columnas v2):
+-------------------+-------+------------------+------------+---------+
|           event_ts|service|cost_usd_increment|genai_tokens|carbon_kg|
+-------------------+-------+------------------+------------+---------+
|2025-08-10 05:05:00|compute|            0.2516|        NULL|  5.38E-4|
|2025-07-30 23:05:00|compute|            6.5886|        NULL|   0.0186|
|2025-08-13 12:54:00|compute|            0.1108|        NULL|  2.74E-4|
|2025-07-11 02:00:00|compute|            0.5922|        NULL|     NULL|
|2025-08-03 01:22:00|compute|            7.7236|        NULL|   0.0226|
|2025-08-13 01:28:00|compute|            1.4459|        NULL|   0.0043|
|2025-07-15 12:52:00|compute|            9.8201|        NULL|     NULL|
|2025-08-18 13:52:00|compute|           11.6419|        NULL|   0.0268|
|2025-07-27 05:44:00|compute|            7.3342|        NULL|     0.02|
|2025-08-15 

## Capa Silver

In [28]:
from pyspark.sql.functions import col, to_date, lower, trim, when, lit, coalesce

print("Procesamiento Silver (Enriquecimiento)...")

# CARGAR DIMENSIONES ESTÁTICAS (Lookups)
# Leemos las tablas maestras de Bronze y las cacheamos en memoria.
print("Cargando dimensiones en memoria...")
try:
    # Organizaciones
    df_orgs = spark.read.parquet(f"{BRONZE_PATH}/customers_orgs") \
        .select("org_id", "org_name", "industry", "plan_tier") \
        .cache()

    # Usuarios
    df_users = spark.read.parquet(f"{BRONZE_PATH}/users") \
        .select("user_id", "email", "role") \
        .cache()

    # Recursos
    df_resources = spark.read.parquet(f"{BRONZE_PATH}/resources") \
        .select("resource_id", "state", "tags_json") \
        .cache()

    print(f"- Dimensiones cargadas: Orgs({df_orgs.count()}), Users({df_users.count()})")

except Exception as e:
    print(f"Error cargando dimensiones: {e}")
    raise e

# LEER STREAM DESDE BRONZE
df_bronze_stream = spark.readStream \
    .format("parquet") \
    .schema(spark.read.parquet(f"{BRONZE_PATH}/usage_events").schema) \
    .load(f"{BRONZE_PATH}/usage_events")

# TRANSFORMACIONES SILVER
# Limpieza Básica y Columnas Derivadas
df_clean = df_bronze_stream \
    .withColumn("usage_date", to_date(col("event_ts"))) \
    .withColumn("region", lower(trim(col("region")))) \
    .withColumn("service", lower(trim(col("service")))) \
    .withColumn("unit", when(col("unit").isNull() & col("value").isNotNull(), "count")
    .otherwise(col("unit")))

# Enriquecimiento (Joins)
# Unimos el evento con la info de la organización y el usuario
df_enriched = df_clean \
    .join(df_orgs, "org_id", "left") \
    .join(df_users, "user_id", "left") \
    .join(df_resources, "resource_id", "left")

# Reglas de Calidad (Valid vs Quarantine)
# Definimos qué es un dato "inválido"
# Costo >= -0.01 se acepta (permitimos pequeños ajustes negativos, pero no errores excesivos)
condicion_valida = (col("cost_usd_increment") >= -0.01) | (col("cost_usd_increment").isNull())

df_silver_valid = df_enriched.filter(condicion_valida)
df_silver_invalid = df_enriched.filter(~condicion_valida) \
    .withColumn("error_reason", lit("Negative Cost Out of Range"))

# ESCRITURA MULTI-STREAM (Valid -> Silver, Invalid -> Quarantine)
print("Iniciando escritura de Streams Silver...")

# Datos Válidos a Silver
query_silver = df_silver_valid \
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/silver_main") \
    .option("path", f"{SILVER_PATH}/usage_events_enriched") \
    .partitionBy("usage_date", "service") \
    .trigger(availableNow=True) \
    .start()

# Datos Inválidos a Quarantine
query_quarantine = df_silver_invalid \
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/silver_quarantine") \
    .option("path", f"{QUARANTINE_PATH}/usage_events_errors") \
    .trigger(availableNow=True) \
    .start()

print("Procesando enriquecimiento y validación...")
query_silver.awaitTermination()
query_quarantine.awaitTermination()

print("Procesamiento Finalizado: Datos en Silver y Quarantine.")

Procesamiento Silver (Enriquecimiento)...
Cargando dimensiones en memoria...
- Dimensiones cargadas: Orgs(80), Users(800)
Iniciando escritura de Streams Silver...
Procesando enriquecimiento y validación...
Procesamiento Finalizado: Datos en Silver y Quarantine.


In [29]:
print("Inspeccionando Silver (Datos Enriquecidos):")

try:
    df_silver_check = spark.read.parquet(f"{SILVER_PATH}/usage_events_enriched")

    print(f"Total registros Silver: {df_silver_check.count()}")

    print("Muestra con cruces (Joins):")
    df_silver_check.select("event_ts", "org_name", "industry", "service", "cost_usd_increment") \
        .show(10, truncate=False)

    print("Inspeccionando Cuarentena (Errores):")
    # Puede que esté vacío si no había costos negativos, lo cual es bueno
    path_quarantine = f"{QUARANTINE_PATH}/usage_events_errors"
    if os.path.exists(path_quarantine):
        try:
            df_quarantine = spark.read.parquet(path_quarantine)
            if df_quarantine.count() > 0:
                print(f"Se encontraron {df_quarantine.count()} registros inválidos.")
                df_quarantine.select("event_id", "cost_usd_increment", "error_reason").show()
            else:
                print("Carpeta de cuarentena vacía (Sin errores graves).")
        except:
             print("Carpeta de cuarentena vacía o sin archivos parquet aún.")
    else:
        print("No se generó carpeta de cuarentena (Datos limpios).")

except Exception as e:
    print(f"Error leyendo Silver: {e}")

Inspeccionando Silver (Datos Enriquecidos):
Total registros Silver: 7214
Muestra con cruces (Joins):
+-------------------+-----------------+-------------+-------+------------------+
|event_ts           |org_name         |industry     |service|cost_usd_increment|
+-------------------+-----------------+-------------+-------+------------------+
|2025-07-05 23:14:00|Vertex Labs 22   |Manufacturing|compute|11.6565           |
|2025-07-05 12:08:00|Nimbus Digital 36|Education    |compute|5.4476            |
|2025-07-05 02:22:00|Gamma Data 15    |Media        |compute|11.0035           |
|2025-07-05 04:37:00|Delta Tech 71    |Education    |compute|9.9519            |
|2025-07-05 14:22:00|Nova Tech 1      |Education    |compute|0.5091            |
|2025-07-05 08:57:00|Nimbus Cloud 76  |Fintech      |compute|9.4997            |
|2025-07-05 21:05:00|Delta Digital 49 |Fintech      |compute|0.2671            |
|2025-07-05 02:16:00|Zenith Cloud 24  |Manufacturing|compute|1.5218            |
|2025-07

## Gold Marts (Agregaciones)

In [30]:
from pyspark.sql.functions import sum, count, avg, col, lit, max as max_col, window, when, expr, abs as spark_abs, round as spark_round
from pyspark.sql.window import Window

print("Construcción de Gold Marts...")

# LIMPIEZA DE CHECKPOINTS Y DATOS
print("Limpiando datos previos...")
paths = [
    f"{CHECKPOINT_PATH}/gold_marts_processor",
    f"{GOLD_PATH}/finops_daily_usage",
    f"{GOLD_PATH}/genai_usage_daily",
    f"{GOLD_PATH}/revenue_monthly",
    f"{GOLD_PATH}/support_tickets_summary",
    f"{GOLD_PATH}/carbon_footprint_daily"
]
for p in paths:
    if os.path.exists(p):
        shutil.rmtree(p)

# MARTs BATCH (Billing y Soporte)
print("\nGenerando Marts Batch (Revenue y Soporte)...")

try:
    # Revenue Monthly (Desde Billing Bronze)
    # Calcular Ingreso Neto = (Total - Créditos + Impuestos)
    # Usamos el CSV original ya que billing es mensual
    df_billing = spark.read.parquet(f"{BRONZE_PATH}/billing_monthly")
    df_revenue_mart = df_billing \
        .withColumnRenamed("month", "billing_month") \
        .withColumn("credits_applied", coalesce(col("credits"), lit(0.0))) \
        .withColumn("gross_total_local", col("subtotal") + col("taxes")) \
        .withColumn("net_revenue_usd", (col("subtotal") + col("taxes") - coalesce(col("credits"), lit(0.0))) * col("exchange_rate_to_usd")) \
        .select(
            "org_id",
            "billing_month",
            "currency",
            col("gross_total_local").alias("total_due_local"),
            "credits_applied",
            col("taxes").alias("tax_amount"),
            col("net_revenue_usd").alias("net_revenue")
        )

    # Escribir a Gold
    df_revenue_mart.write.mode("overwrite").parquet(f"{GOLD_PATH}/revenue_monthly")
    print(f"- revenue_monthly creado. Registros: {df_revenue_mart.count()}")

    # Support Summary (Desde Tickets Bronze)
    # Agrupar por Org, Fecha y Severidad
    df_tickets = spark.read.parquet(f"{BRONZE_PATH}/support_tickets")
    df_support_mart = df_tickets \
        .groupBy("org_id", to_date("created_at").alias("ticket_date"), "severity") \
        .agg(
            count("ticket_id").alias("total_tickets"),
            avg("csat").alias("avg_csat"),
            sum(col("sla_breached").cast("int")).alias("sla_breaches_count")
        )

    # Escribir a Gold
    df_support_mart.write.mode("overwrite").parquet(f"{GOLD_PATH}/support_tickets_summary")
    print(f"- support_tickets_summary creado. Registros: {df_support_mart.count()}")

except Exception as e:
    print(f"Error en Marts Batch: {e}")


# MARTs STREAMING (FinOps y GenAI, Carbono)
print("\nConfigurando Marts Streaming (FinOps, GenAI, y Carbono)...")

# Leemos desde SILVER
df_silver_stream = spark.readStream \
    .format("parquet") \
    .schema(spark.read.parquet(f"{SILVER_PATH}/usage_events_enriched").schema) \
    .load(f"{SILVER_PATH}/usage_events_enriched")

# Función que procesará cada micro-batch
def process_gold_marts(df_batch, batch_id):
    df_batch.cache()

    if df_batch.count() > 0:
        # MART 1: FinOps Daily Usage con Detección de Anomalías
        # Agregamos por Org, Fecha y Servicio
        df_finops = df_batch \
            .groupBy("org_id", "usage_date", "service") \
            .agg(
                sum("cost_usd_increment").alias("total_cost_usd"),
                count("event_id").alias("total_requests"),
                sum("genai_tokens").alias("total_tokens"),
                sum("carbon_kg").alias("total_carbon_kg")
            )

        # Calcular MAD (Median Absolute Deviation) para detección de anomalías
        # Calculamos anomalías usando Window Functions sobre el resultado agregado

        # Calcular Mediana
        df_finops_stats = df_finops \
            .withColumn("median_cost", expr("percentile_approx(total_cost_usd, 0.5) OVER (PARTITION BY org_id, service)")) \
            .withColumn("cost_diff", spark_abs(col("total_cost_usd") - col("median_cost")))

        # Calcular MAD (Mediana de la diferencia absoluta)
        df_finops_final = df_finops_stats \
            .withColumn("mad", expr("percentile_approx(cost_diff, 0.5) OVER (PARTITION BY org_id, service)")) \
            .withColumn("anomaly_score",
                        when(col("mad") > 0, col("cost_diff") / col("mad")).otherwise(0.0)) \
            .withColumn("is_anomaly", col("anomaly_score") > 3.5) \
            .drop("median_cost", "cost_diff", "mad")

        df_finops_final.write.mode("append").parquet(f"{GOLD_PATH}/finops_daily_usage")

        # MART 2: GenAI Specific, solo si hay datos de GenAI
        df_genai = df_batch.filter(col("service") == "genai") \
            .groupBy("org_id", "usage_date") \
            .agg(
                sum("genai_tokens").alias("total_tokens"),
                count("event_id").alias("total_requests"),
                sum("cost_usd_increment").alias("total_cost_usd"),
                avg("genai_tokens").alias("avg_tokens_per_request")
            )

        if df_genai.count() > 0:
            df_genai.write.mode("append").parquet(f"{GOLD_PATH}/genai_usage_daily")

        # MART 3: Carbon Footprint Daily (Nueva agregación)
        df_carbon = df_batch \
            .filter(col("carbon_kg").isNotNull()) \
            .groupBy("org_id", "usage_date") \
            .agg(
                sum("carbon_kg").alias("total_carbon_kg"),
                count("event_id").alias("events_with_carbon_data")
            )

        if df_carbon.count() > 0:
            df_carbon.write.mode("append").parquet(f"{GOLD_PATH}/carbon_footprint_daily")

    df_batch.unpersist()

# Stream con trigger AvailableNow (Procesa lo pendiente y termina)
query_gold = df_silver_stream.writeStream \
    .foreachBatch(process_gold_marts) \
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/gold_marts_processor") \
    .trigger(availableNow=True) \
    .start()

print("Calculando agregaciones Gold...")
query_gold.awaitTermination()
print("Marts generados!")

Construcción de Gold Marts...
Limpiando datos previos...

Generando Marts Batch (Revenue y Soporte)...
- revenue_monthly creado. Registros: 240
- support_tickets_summary creado. Registros: 984

Configurando Marts Streaming (FinOps, GenAI, y Carbono)...
Calculando agregaciones Gold...
Marts generados!


In [31]:
from pyspark.sql.functions import sum, col, date_sub, current_date, rank, dense_rank, expr, when, lit
from pyspark.sql.window import Window

print("Generando Mart Top-N Servicios...")

try:
    df_finops_all = spark.read.parquet(f"{GOLD_PATH}/finops_daily_usage")
    max_date = df_finops_all.agg({"usage_date": "max"}).collect()[0][0]
    print(f"   Fecha de corte para análisis: {max_date}")

    # CALCULAR VENTANAS TEMPORALES
    df_windows = df_finops_all \
        .withColumn("days_ago", expr(f"datediff(date'{max_date}', usage_date)")) \
        .withColumn("cost_1d", when(col("days_ago") == 0, col("total_cost_usd")).otherwise(0.0)) \
        .withColumn("cost_7d", when(col("days_ago") < 7, col("total_cost_usd")).otherwise(0.0)) \
        .withColumn("cost_14d", when(col("days_ago") < 14, col("total_cost_usd")).otherwise(0.0))

    # AGREGAR Y RANKEAR
    df_cost_summary = df_windows \
        .groupBy("org_id", "service") \
        .agg(
            sum("cost_1d").alias("cost_1d"),
            sum("cost_7d").alias("cost_7d"),
            sum("cost_14d").alias("cost_14d")
        ) \
        .withColumn("summary_date", lit(max_date))

    # Ranking por Org basado en los últimos 14 días
    window_ranking = Window.partitionBy("org_id").orderBy(col("cost_14d").desc())

    df_cost_ranked = df_cost_summary \
        .withColumn("rank_14d", dense_rank().over(window_ranking)) \
        .filter(col("rank_14d") <= 10)  # Top 10

    # Escribir en Gold
    df_cost_ranked.write.mode("overwrite").parquet(f"{GOLD_PATH}/org_service_cost_summary")
    print(f"- org_service_cost_summary creado. Filas: {df_cost_ranked.count()}")

except Exception as e:
    print(f"Error: {e}")

Generando Mart Top-N Servicios...
   Fecha de corte para análisis: 2025-08-31
- org_service_cost_summary creado. Filas: 262


In [32]:
print("Inspeccionando Gold Marts (KPIs de Negocio):")

try:
    # FinOps Check
    print("\nFinOps Daily Usage (Costos por Día/Servicio):")
    df_finops = spark.read.parquet(f"{GOLD_PATH}/finops_daily_usage")
    df_finops.orderBy(col("total_cost_usd").desc()).show(10)
    print(f"- Total registros FinOps: {df_finops.count()}")
    print("- Top 10 por costo:")
    df_finops.orderBy(col("total_cost_usd").desc()).show(10)

    # Mostrar anomalías detectadas
    df_anomalies = df_finops.filter(col("is_anomaly") == True)
    anomaly_count = df_anomalies.count()
    if anomaly_count > 0:
        print(f"\nANOMALÍAS DETECTADAS: {anomaly_count} registros")
        df_anomalies.select("org_id", "usage_date", "service", "total_cost_usd", "anomaly_score") \
            .orderBy(col("anomaly_score").desc()) \
            .show(10, truncate=False)
    else:
        print("No se detectaron anomalías significativas (MAD score > 3.5)")

    # Revenue Check
    print("\nMonthly Revenue (Facturación):")
    df_rev = spark.read.parquet(f"{GOLD_PATH}/revenue_monthly")
    df_rev.show(10)

    # GenAI Check
    if os.path.exists(f"{GOLD_PATH}/genai_usage_daily"):
        print("\nGenAI Usage (Tokens):")
        df_genai = spark.read.parquet(f"{GOLD_PATH}/genai_usage_daily")
        print(f"- Total registros GenAI: {df_genai.count()}")
        df_genai.show(10)
    else:
        print("\nGenAI Usage: No se encontraron eventos de GenAI en este lote.")

    # Carbon Footprint Check
    if os.path.exists(f"{GOLD_PATH}/carbon_footprint_daily"):
        print("\nCarbon Footprint (Huella de Carbono):")
        df_carbon = spark.read.parquet(f"{GOLD_PATH}/carbon_footprint_daily")
        print(f"- Total registros Carbon: {df_carbon.count()}")
        df_carbon.orderBy(col("total_carbon_kg").desc()).show(10)

        total_carbon = df_carbon.agg({"total_carbon_kg": "sum"}).collect()[0][0]
        print(f"- Huella de Carbono Total: {total_carbon:.2f} kg CO2")
    else:
        print("\nCarbon Footprint: No hay datos de carbono en v1 events")

    # Top-N Services Check
    if os.path.exists(f"{GOLD_PATH}/org_service_cost_summary"):
        print("\nTop-N Servicios por Costo (Ventanas Temporales):")
        df_top_services = spark.read.parquet(f"{GOLD_PATH}/org_service_cost_summary")
        print(f"- Total registros: {df_top_services.count()}")
        df_top_services.filter(col("rank_14d") <= 5) \
            .select("org_id", "service", "rank_14d", "cost_14d", "cost_7d", "cost_1d") \
            .orderBy("org_id", "rank_14d") \
            .show(20, truncate=False)

except Exception as e:
    print(f"Error leyendo Gold: {e}")

Inspeccionando Gold Marts (KPIs de Negocio):

FinOps Daily Usage (Costos por Día/Servicio):
+------------+----------+--------+------------------+--------------+------------+---------------+------------------+----------+
|      org_id|usage_date| service|    total_cost_usd|total_requests|total_tokens|total_carbon_kg|     anomaly_score|is_anomaly|
+------------+----------+--------+------------------+--------------+------------+---------------+------------------+----------+
|org_c11ertj5|2025-07-11|   genai|          317.4308|             1|        NULL|           NULL| 297.9272059515962|      true|
|org_tvhhpbmy|2025-07-04|   genai|           300.396|             1|        NULL|           NULL|128.17480111803914|      true|
|org_kdgigatj|2025-07-17| compute|          211.3419|             5|        NULL|           NULL| 44.12903990858326|      true|
|org_okep7y6w|2025-07-03| compute|          201.4929|             1|        NULL|           NULL| 36.19415868291322|      true|
|org_teiyzco

## Carga de datos a Cassandra

In [33]:
from pyspark.sql.functions import col, to_date
import os

print("Inciando Carga de Datos a Cassandra...")

# Función auxiliar de carga segura
def load_to_cassandra(parquet_path, table_name, transformation_func=None):
    if not os.path.exists(parquet_path):
        print(f"{table_name}: No existe el archivo en Gold para esta tabla")
        return

    try:
        print(f"- Leyendo {table_name} desde Gold...")
        df = spark.read.parquet(parquet_path)

        # Aplicar casting de tipos si es necesario
        if transformation_func:
            df = transformation_func(df)

        print(f"- Escribiendo registros en AstraDB...")
        df.write \
            .format("org.apache.spark.sql.cassandra") \
            .mode("append") \
            .options(table=table_name, keyspace="cloud_analytics") \
            .save()

        count = df.count()
        print(f"{table_name}: Se escribieron {count} registros\n")

    except Exception as e:
        print(f"ERROR en {table_name}: {e}")

# Casting
def tr_finops(df):
    return df.select(
        col("org_id").cast("string"),
        col("usage_date").cast("date"),
        col("service").cast("string"),
        col("total_cost_usd").cast("double"),
        col("total_requests").cast("long"),
        col("total_tokens").cast("long"),
        col("total_carbon_kg").cast("double"),
        col("is_anomaly").cast("boolean"),
        col("anomaly_score").cast("double")
    )

def tr_cost_summary(df):
    return df.select(
        col("org_id").cast("string"),
        col("summary_date").cast("date"),
        col("service").cast("string"),
        col("cost_14d").cast("double"),
        col("cost_7d").cast("double"),
        col("cost_1d").cast("double"),
        col("rank_14d").cast("int")
    )

def tr_support(df):
    return df.select(
        col("org_id").cast("string"),
        col("ticket_date").cast("date"),
        col("severity").cast("string"),
        col("total_tickets").cast("int"),
        col("avg_csat").cast("double"),
        col("sla_breaches_count").cast("int")
    )

def tr_revenue(df):
    return df.select(
        col("org_id").cast("string"),
        col("billing_month").cast("string"),
        col("currency").cast("string"),
        col("total_due_local").cast("double"),
        col("credits_applied").cast("double"),
        col("tax_amount").cast("double"),
        col("net_revenue").cast("double")
    )

def tr_genai(df):
    return df.select(
        col("org_id").cast("string"),
        col("usage_date").cast("date"),
        col("total_tokens").cast("long"),
        col("total_requests").cast("int"),
        col("total_cost_usd").cast("double"),
        col("avg_tokens_per_request").cast("int")
    )

def tr_carbon(df):
    return df.select(
        col("org_id").cast("string"),
        col("usage_date").cast("date"),
        col("total_carbon_kg").cast("double"),
        col("events_with_carbon_data").cast("int")
    )

load_to_cassandra(f"{GOLD_PATH}/finops_daily_usage", "finops_daily_usage", tr_finops)
load_to_cassandra(f"{GOLD_PATH}/org_service_cost_summary", "org_service_cost_summary", tr_cost_summary)
load_to_cassandra(f"{GOLD_PATH}/support_tickets_summary", "support_tickets_summary", tr_support)
load_to_cassandra(f"{GOLD_PATH}/revenue_monthly", "revenue_monthly", tr_revenue)
load_to_cassandra(f"{GOLD_PATH}/genai_usage_daily", "genai_usage_daily", tr_genai)
load_to_cassandra(f"{GOLD_PATH}/carbon_footprint_daily", "carbon_footprint_daily", tr_carbon)

print("\nDatos cargados Exitosamente!")

Inciando Carga de Datos a Cassandra...
- Leyendo finops_daily_usage desde Gold...
- Escribiendo registros en AstraDB...
finops_daily_usage: Se escribieron 5436 registros

- Leyendo org_service_cost_summary desde Gold...
- Escribiendo registros en AstraDB...
org_service_cost_summary: Se escribieron 262 registros

- Leyendo support_tickets_summary desde Gold...
- Escribiendo registros en AstraDB...
support_tickets_summary: Se escribieron 984 registros

- Leyendo revenue_monthly desde Gold...
- Escribiendo registros en AstraDB...
revenue_monthly: Se escribieron 240 registros

- Leyendo genai_usage_daily desde Gold...
- Escribiendo registros en AstraDB...
genai_usage_daily: Se escribieron 545 registros

- Leyendo carbon_footprint_daily desde Gold...
- Escribiendo registros en AstraDB...
carbon_footprint_daily: Se escribieron 2604 registros


Datos cargados Exitosamente!


## Data Reconciliation

In [34]:
print("=== Data Reconciliation ===\n")

marts = [
    "finops_daily_usage",
    "org_service_cost_summary",
    "support_tickets_summary",
    "revenue_monthly",
    "genai_usage_daily",
    "carbon_footprint_daily"
]

results = []
for mart in marts:
    parquet_path = f"{GOLD_PATH}/{mart}"

    if not os.path.exists(parquet_path):
        continue

    # Comparar conteo Parquet vs. Cassandra
    parquet_count = spark.read.parquet(parquet_path).count()
    cassandra_count = spark.sql(f"SELECT COUNT(*) as cnt FROM myCatalog.cloud_analytics.{mart}").collect()[0]['cnt']

    match = "[OK]" if parquet_count == cassandra_count else "[MISMATCH]"
    results.append({
        'Tabla': mart,
        'Parquet': parquet_count,
        'Cassandra': cassandra_count,
        'Match': match
    })

    print(f"{match} {mart}: Parquet={parquet_count:,}, Cassandra={cassandra_count:,}")

if all(r['Match'] == '[OK]' for r in results):
    print("\n[OK] Todos los datos cargados correctamente!")
else:
    print("\n[WARN] Hay discrepancias en algunos marts")

=== Data Reconciliation ===

[OK] finops_daily_usage: Parquet=5,436, Cassandra=5,436
[OK] org_service_cost_summary: Parquet=262, Cassandra=262
[OK] support_tickets_summary: Parquet=984, Cassandra=984
[OK] revenue_monthly: Parquet=240, Cassandra=240
[OK] genai_usage_daily: Parquet=545, Cassandra=545
[OK] carbon_footprint_daily: Parquet=2,604, Cassandra=2,604

[OK] Todos los datos cargados correctamente!


# Consultas sobre AstraDB

In [35]:
from pyspark.sql.functions import col

print("=== CONSULTAAS SOBRE ASTRADB ===\n")

SNAPSHOT_DATE = "2025-08-31"

# Usamos una org de ejemplo
sample_org = spark.sql("SELECT DISTINCT org_id FROM myCatalog.cloud_analytics.finops_daily_usage LIMIT 1").collect()[0]['org_id']
print(f"Usando org_id: {sample_org}\n")

# ================================================
# Costos y requests diarios por org y servicio
# ================================================
print("=" * 80)
print("Costos y requests diarios por org/servicio (Ultimos 30 dias)")
print("=" * 80)

query1 = f"""
SELECT
    org_id,
    usage_date,
    service,
    total_cost_usd,
    total_requests,
    is_anomaly,
    anomaly_score
FROM myCatalog.cloud_analytics.finops_daily_usage
WHERE org_id = '{sample_org}'
  AND usage_date >= date_sub('{SNAPSHOT_DATE}', 30)
ORDER BY usage_date DESC, total_cost_usd DESC
"""

df_q1 = spark.sql(query1)
print(f"\nResultados: {df_q1.count()} registros")
df_q1.show(20, truncate=False)

# ================================================
# Top-N servicios por costo acumulado (14 días)
# ================================================
print("\n" + "=" * 80)
print("Top-10 servicios por costo acumulado (últimos 14 días)")
print("=" * 80)

# fecha más reciente del summary
max_summary_date = spark.sql("""
    SELECT MAX(summary_date) as max_date
    FROM myCatalog.cloud_analytics.org_service_cost_summary
""").collect()[0]['max_date']

query2 = f"""
SELECT
    org_id,
    service,
    rank_14d,
    cost_14d,
    cost_7d,
    cost_1d,
    ROUND((cost_7d / cost_14d) * 100, 2) as cost_7d_pct_of_14d
FROM myCatalog.cloud_analytics.org_service_cost_summary
WHERE org_id = '{sample_org}'
  AND summary_date = '{max_summary_date}'
  AND rank_14d <= 10
ORDER BY rank_14d ASC
"""

df_q2 = spark.sql(query2)
print(f"\nResultados: {df_q2.count()} registros")
df_q2.show(10, truncate=False)

# ================================================
# Evolución de tickets críticos y SLA breach
# ================================================
print("\n" + "=" * 80)
print("Evolución de tickets críticos y SLA breach")
print("=" * 80)

query3 = f"""
SELECT
    org_id,
    ticket_date,
    severity,
    total_tickets,
    sla_breaches_count,
    ROUND((sla_breaches_count / total_tickets) * 100, 2) as sla_breach_rate,
    avg_csat
FROM myCatalog.cloud_analytics.support_tickets_summary
WHERE org_id = '{sample_org}'
  AND severity IN ('critical', 'high')
  AND ticket_date >= date_sub('{SNAPSHOT_DATE}', 30)
ORDER BY ticket_date DESC, severity
"""

df_q3 = spark.sql(query3)
print(f"\nResultados: {df_q3.count()} registros")
df_q3.show(30, truncate=False)

# ================================================
# Revenue mensual con créditos/impuestos
# ================================================
print("\n" + "=" * 80)
print("Revenue mensual normalizado a USD")
print("=" * 80)

query4 = f"""
SELECT
    org_id,
    billing_month,
    currency,
    total_due_local,
    credits_applied,
    tax_amount,
    net_revenue,
    ROUND((credits_applied / total_due_local) * 100, 2) as credits_pct,
    ROUND((tax_amount / total_due_local) * 100, 2) as tax_pct
FROM myCatalog.cloud_analytics.revenue_monthly
WHERE org_id = '{sample_org}'
ORDER BY billing_month DESC
"""

df_q4 = spark.sql(query4)
print(f"\nResultados: {df_q4.count()} registros")
df_q4.show(truncate=False)

# ================================================
# Tokens GenAI y costo estimado por día
# ================================================
print("\n" + "=" * 80)
print("Uso de GenAI - Tokens y costo diario")
print("=" * 80)

query5 = f"""
SELECT
    org_id,
    usage_date,
    total_tokens,
    total_requests,
    total_cost_usd,
    avg_tokens_per_request,
    ROUND(total_cost_usd / total_tokens * 1000000, 4) as cost_per_million_tokens
FROM myCatalog.cloud_analytics.genai_usage_daily
WHERE org_id = '{sample_org}'
  AND usage_date >= date_sub('{SNAPSHOT_DATE}', 30)
ORDER BY usage_date DESC
"""

df_q5 = spark.sql(query5)
print(f"\nResultados: {df_q5.count()} registros")
if df_q5.count() > 0:
    df_q5.show(30, truncate=False)

    print("\nEstadísticas agregadas de GenAI:")
    df_q5.selectExpr(
        "SUM(total_tokens) as total_tokens_30d",
        "SUM(total_requests) as total_requests_30d",
        "SUM(total_cost_usd) as total_cost_30d",
        "AVG(avg_tokens_per_request) as avg_tokens_per_request"
    ).show(truncate=False)
else:
    print("No se encontraron datos de GenAI para esta organización")

=== CONSULTAAS SOBRE ASTRADB ===

Usando org_id: org_0lzjjege

Costos y requests diarios por org/servicio (Ultimos 30 dias)

Resultados: 34 registros
+------------+----------+----------+--------------------+--------------+----------+-------------------+
|org_id      |usage_date|service   |total_cost_usd      |total_requests|is_anomaly|anomaly_score      |
+------------+----------+----------+--------------------+--------------+----------+-------------------+
|org_0lzjjege|2025-08-31|compute   |0.240900000000000000|1             |false     |0.8698144104803492 |
|org_0lzjjege|2025-08-31|networking|0.118300000000000000|1             |false     |0.7996146435452792 |
|org_0lzjjege|2025-08-30|storage   |2.788800000000000000|1             |false     |0.7486153498361027 |
|org_0lzjjege|2025-08-30|networking|0.161600000000000000|2             |false     |0.5910404624277457 |
|org_0lzjjege|2025-08-29|storage   |2.639700000000000000|1             |false     |0.5800836441731659 |
|org_0lzjjege|2025

In [38]:
# ================================================
# Huella de Carbono por Organización
# ================================================
print("=" * 80)
print("Huella de Carbono por Organización")
print("=" * 80)

SNAPSHOT_DATE = "2025-08-31"

# Usamos una org de ejemplo
sample_org = spark.sql("SELECT DISTINCT org_id FROM myCatalog.cloud_analytics.finops_daily_usage LIMIT 1").collect()[0]['org_id']
print(f"Usando org_id: {sample_org}\n")

query_carbon = f"""
SELECT
    org_id,
    usage_date,
    total_carbon_kg,
    events_with_carbon_data,
    ROUND(total_carbon_kg / events_with_carbon_data, 4) as avg_carbon_per_event
FROM myCatalog.cloud_analytics.carbon_footprint_daily
WHERE org_id = '{sample_org}'
  AND usage_date >= date_sub('{SNAPSHOT_DATE}', 30)
ORDER BY usage_date DESC
"""

df_carbon = spark.sql(query_carbon)
if df_carbon.count() > 0:
    print(f"\nResultados: {df_carbon.count()} registros")
    df_carbon.show(30, truncate=False)

    total_carbon = df_carbon.agg({"total_carbon_kg": "sum"}).collect()[0][0]
    print(f"\nHuella de Carbono Total (30 días): {total_carbon:.2f} kg CO2")
else:
    print("\nNo hay datos de carbono (solo en schema v2)")

Huella de Carbono por Organización
Usando org_id: org_c11ertj5


Resultados: 23 registros
+------------+----------+--------------------+-----------------------+--------------------+
|org_id      |usage_date|total_carbon_kg     |events_with_carbon_data|avg_carbon_per_event|
+------------+----------+--------------------+-----------------------+--------------------+
|org_c11ertj5|2025-08-30|0.002689000000000000|1                      |0.0027              |
|org_c11ertj5|2025-08-29|0.050700000000000000|3                      |0.0169              |
|org_c11ertj5|2025-08-28|0.024200000000000000|1                      |0.0242              |
|org_c11ertj5|2025-08-27|0.023000000000000000|1                      |0.0230              |
|org_c11ertj5|2025-08-26|0.000000000000000000|1                      |0.0000              |
|org_c11ertj5|2025-08-24|0.025200000000000000|1                      |0.0252              |
|org_c11ertj5|2025-08-20|0.021816000000000000|2                      |0.0109      